# Lab: kinematic models of glacier response

```{note}
This lab accompanies {doc}`glacier-variations`. Every equation here is derived in that chapter, and the lab's job is to make them run, wander, and lag in front of you.
```

A glacier terminus is a low-pass filter applied to climate. In this lab you will build that filter in three increasingly faithful versions and force each one with steps, noise, and ramps. By the end you should be able to

- integrate the one-stage response equation and verify the 63% and 86% checkpoints of the exponential step response,
- chain three linear reservoirs into the Roe-Baker model and see why its sigmoidal onset is the realistic one,
- force both models with white-noise mass balance and compare the red-noise wanderings they produce,
- run a warming ramp and measure the lag and the committed retreat for two real glaciers,
- as an optional extension, build the Robel two-stage marine glacier model and watch its stability flip with the sign of the bed slope.

Pure numpy and matplotlib throughout.

## The one-stage model

The chapter reduced a glacier to a single reservoir. A persistent mass-balance perturbation $\dot b'$ fills it, ablation over the advancing terminus wedge drains it, and the length perturbation $L'$ obeys

$$ \frac{dL'}{dt} + \frac{L'}{\tau} = \dot b'\,\frac{\bar Y}{Y_t}\,\frac{L_0}{H_0}, \qquad \tau = \frac{H}{|\dot b_t|}, $$

with $\tau$ the Jóhannesson-Raymond-Waddington response time, the ice thickness over the (magnitude of the) terminus balance rate. The chapter wrote this timescale $t_r$; we switch to $\tau$ here to match the Roe-Baker notation that arrives below, and we set the width ratio $\bar Y / Y_t = 1$ throughout so the geometry factor reduces to $L_0/H_0$. For a step change in balance the solution is exponential, reaching 63% of the new equilibrium after one $\tau$ and 86% after two.

Two real glaciers from the chapter will follow us through the whole lab.

| Glacier | $H$ (m) | $\dot b_t$ (m yr$^{-1}$) | $\tau$ (yr) | $L_0$ (km) |
|---|---|---|---|---|
| Nisqually (Mount Rainier) | 96 | $-8.0$ | 12 | 7.0 |
| South Cascade | 200 | $-5.4$ | 37 | 3.4 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

glaciers = {
    'Nisqually':     dict(H=96.0,  bt=-8.0, L0=7000.0),
    'South Cascade': dict(H=200.0, bt=-5.4, L0=3400.0),
}

for name, g in glaciers.items():
    g['tau'] = g['H'] / abs(g['bt'])
    g['geom'] = g['L0'] / g['H']      # the factor (Ybar/Yt)(L0/H0) with Ybar/Yt = 1
    print(f"{name:14s}  tau = {g['tau']:5.1f} yr   L0/H0 = {g['geom']:5.1f}")

**Task 1:** Complete the forward-Euler integrator below, which takes an arbitrary time series of balance perturbations. Apply a step of $\dot b' = +0.5\ \mathrm{m\,yr^{-1}}$ switched on at $t = 0$ to both glaciers, plot $L'(t)/\Delta L_{\mathrm{eq}}$ against $t/\tau$, and confirm that the two curves collapse onto a single exponential. Print the fraction of the equilibrium response reached at $t = \tau$ and $t = 2\tau$, which should land near 0.63 and 0.86. The equilibrium response itself is $\Delta L_{\mathrm{eq}} = \tau\,\dot b'\,L_0/H_0$; compute it for each glacier and notice how a half-meter balance change moves a terminus by hundreds of meters.

In [ ]:
def one_stage(t, bdot, tau, geom):
    """Integrate dL'/dt + L'/tau = bdot(t) * geom with forward Euler.

    t    : uniform time array, yr
    bdot : array of balance perturbations at the times t, m/yr
    tau  : response time, yr
    geom : the geometric factor L0/H0 (dimensionless)
    """
    L = np.zeros_like(t)
    dt = t[1] - t[0]
    for i in range(len(t) - 1):
        # YOUR CODE HERE: update L[i+1] from L[i]
        pass
    return L


dt = 0.1
t = np.arange(0.0, 200.0 + dt, dt)
bdot_step = np.full_like(t, 0.5)

fig, ax = plt.subplots(figsize=(6, 4))
for name, g in glaciers.items():
    # L = one_stage(t, bdot_step, g['tau'], g['geom'])
    # dL_eq = g['tau'] * 0.5 * g['geom']
    # ax.plot(t / g['tau'], L / dL_eq, lw=2, label=name)
    # YOUR CODE HERE: print dL_eq and the fractions reached at tau and 2*tau
    pass
ax.axhline(1 - np.exp(-1), color='gray', ls=':')
ax.axhline(1 - np.exp(-2), color='gray', ls=':')
ax.set_xlabel('t / tau')
ax.set_ylabel('fraction of equilibrium response')
ax.legend()
plt.show()

## The three-stage model

The one-stage model has a flaw visible in your Task 1 plot. Its terminus starts moving at $t = 0$ at its maximum rate, as if a balance perturbation falling on the accumulation zone could move the terminus the same year. A real glacier responds in sequence. The interior thickens first, the extra thickness drives extra flux toward the terminus, and only when that flux arrives does the length change in earnest. Roe and Baker chained three linear reservoirs to capture exactly this,

$$ \frac{dh'}{dt}+\frac{h'}{\epsilon\tau}=\dot b', \qquad \frac{dF'}{dt}+\frac{F'}{\epsilon\tau}=\frac{L_0\,h'}{(\epsilon\tau)^{2}}, \qquad \frac{dL'}{dt}+\frac{L'}{\epsilon\tau}=\frac{F'}{\epsilon H_0}, $$

with the same $\tau$ as before and $\epsilon = 1/\sqrt{3}$ chosen so that the chain preserves the equilibrium response and the overall memory of the one-stage model. The step response, derived in the chapter, is sigmoidal,

$$ L'(t) = L'_{\mathrm{eq}}\left[1 - e^{-t/\epsilon\tau}\left(1 + \frac{t}{\epsilon\tau} + \frac{1}{2}\left(\frac{t}{\epsilon\tau}\right)^{2}\right)\right], $$

flat at first while the anomaly works its way down-glacier, then catching up.

**Task 2:** Complete `three_stage` below, integrating the three equations with forward Euler. Apply the same $+0.5\ \mathrm{m\,yr^{-1}}$ step to South Cascade, overlay the one-stage response, your numerical three-stage response, and the analytic sigmoid, and check that all reach the same equilibrium. Print the fraction of equilibrium reached at $t = \tau, 2\tau, 3\tau$ for both models. Which model would you rather have fit an e-folding time to?

In [ ]:
EPS = 1.0 / np.sqrt(3.0)


def three_stage(t, bdot, tau, L0, H0):
    """Roe-Baker three-stage model, forward Euler.

    Returns the length perturbation L' (m) at the times t (yr).
    State variables: h (interior thickness anomaly, m),
    F (terminus flux anomaly, m^2/yr), L (length anomaly, m).
    """
    h = np.zeros_like(t)
    F = np.zeros_like(t)
    L = np.zeros_like(t)
    dt = t[1] - t[0]
    et = EPS * tau
    for i in range(len(t) - 1):
        # YOUR CODE HERE: update h[i+1], F[i+1], L[i+1] from the
        # three chained equations
        pass
    return L


g = glaciers['South Cascade']
tau, L0, H0 = g['tau'], g['L0'], g['H']
dL_eq = tau * 0.5 * L0 / H0

et = EPS * tau
x = t / et
L_analytic = dL_eq * (1 - np.exp(-x) * (1 + x + 0.5 * x**2))

fig, ax = plt.subplots(figsize=(6, 4))
# L1 = one_stage(t, bdot_step, tau, L0 / H0)
# L3 = three_stage(t, bdot_step, tau, L0, H0)
# ax.plot(t, L1, lw=2, label='one stage')
# ax.plot(t, L3, lw=2, label='three stage (numerical)')
ax.plot(t, L_analytic, 'k--', lw=1, label='three stage (analytic)')
ax.set_xlabel('time (yr)')
ax.set_ylabel("L' (m)")
ax.legend()
plt.show()

# YOUR CODE HERE: print the fraction of dL_eq reached at tau, 2*tau,
# 3*tau for both models

## Stochastic forcing

Interannual mass-balance variability is essentially white noise, a meter or so of water equivalent from year to year with little memory. The glacier integrates it. Run white noise through either model and the terminus performs slow, red-noise wanderings over decades, with no climate trend anywhere in the forcing. For the one-stage model forced annually ($\Delta t = 1$ yr) with noise of standard deviation $\sigma_b$, Roe and Baker give the closed-form variance

$$ \sigma_L^2 = \frac{\tau\,\Delta t}{2}\left(\frac{L_0}{H_0}\right)^{2}\sigma_b^2, $$

and the three-stage model, which filters high frequencies more aggressively, wanders with roughly three quarters of that variance for glaciers like these.

**Task 3:** Generate 10,000 years of annual balance anomalies drawn from $N(0, \sigma_b^2)$ with $\sigma_b = 1.0\ \mathrm{m\,yr^{-1}}$, run both models for South Cascade with $\Delta t = 1$ yr, and plot a 2,000-year window of the two length records. Compute the standard deviation of each, compare the one-stage value with the analytic prediction, and report the three-stage to one-stage variance ratio. Then answer the question this experiment exists to pose. If you were handed a single length record showing a 300-m advance over 50 years, could you conclude the climate had changed?

In [ ]:
rng = np.random.default_rng(seed=431)

dt_n = 1.0
t_n = np.arange(0.0, 10000.0, dt_n)
sigma_b = 1.0
bdot_noise = rng.normal(0.0, sigma_b, size=t_n.size)

g = glaciers['South Cascade']

# L1 = one_stage(t_n, bdot_noise, g['tau'], g['geom'])
# L3 = three_stage(t_n, bdot_noise, g['tau'], g['L0'], g['H'])

fig, ax = plt.subplots(figsize=(9, 4))
window = slice(4000, 6000)
# ax.plot(t_n[window], L1[window], lw=1, label='one stage')
# ax.plot(t_n[window], L3[window], lw=1.5, label='three stage')
ax.axhline(0.0, color='gray', lw=0.5)
ax.set_xlabel('time (yr)')
ax.set_ylabel("L' (m)")
ax.legend()
plt.show()

sigma_L_analytic = np.sqrt(g['tau'] * dt_n / 2.0) * g['geom'] * sigma_b
print(f'analytic one-stage sigma_L = {sigma_L_analytic:.0f} m')
# YOUR CODE HERE: print the numerical sigma_L for both models (discard
# the first ~500 yr of spin-up) and the three-stage/one-stage variance ratio

## Ramp forcing and committed retreat

Now force the models with something resembling the last century. A warming trend $T'(t) = \dot T t$ maps onto mass balance through the melt factor $\mu$, so $\dot b'(t) = -\mu\,T'(t)$, and we take $\mu = 0.65\ \mathrm{m\,yr^{-1}\,K^{-1}}$ from Roe and Baker. Apply $\dot T = 0.02\ \mathrm{K\,yr^{-1}}$ for 100 years, then hold the climate fixed at its new, 2-degree-warmer state for another 150 years.

At every instant there is an equilibrium length the glacier would have if it could keep up, $L'_{\mathrm{eq}}(t) = \tau\,\dot b'(t)\,L_0/H_0$. During the ramp the terminus chases this target and falls behind it by an amount that grows with $\tau$, and when the warming stops the chase continues. The gap between actual and equilibrium length at the moment the climate stabilizes is the **committed retreat**, the retreat already purchased but not yet delivered.

**Task 4:** Run the three-stage model through the ramp-and-hold scenario for both glaciers. For each, plot $L'(t)$ with $L'_{\mathrm{eq}}(t)$ dashed, and print the committed retreat at year 100. Then answer two questions from the plot. Which glacier tracks its climate, and which carries a bank of unrealized retreat? And how many years after the warming stops does each glacier take to close 90% of its gap?

In [ ]:
mu = 0.65        # melt factor, m/yr per K
Tdot = 0.02      # warming rate, K/yr
t_ramp = 100.0   # duration of the ramp, yr

dt_r = 0.1
t_r = np.arange(0.0, 250.0 + dt_r, dt_r)
Tprime = np.where(t_r < t_ramp, Tdot * t_r, Tdot * t_ramp)
bdot_ramp = -mu * Tprime

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, (name, g) in zip(axes, glaciers.items()):
    # L = three_stage(t_r, bdot_ramp, g['tau'], g['L0'], g['H'])
    # L_eq = g['tau'] * bdot_ramp * g['geom']
    # ax.plot(t_r, L, lw=2, label='terminus')
    # ax.plot(t_r, L_eq, 'k--', lw=1, label='equilibrium')
    # YOUR CODE HERE: committed retreat = L_eq - L at t = 100 yr; print it,
    # and find when the remaining gap first falls below 10% of that value
    ax.axvline(t_ramp, color='gray', ls=':')
    ax.set_title(name)
    ax.set_xlabel('time (yr)')
    ax.set_ylabel("L' (m)")
    ax.legend()
plt.tight_layout()
plt.show()

## Optional extension to marine glaciers

For a marine-terminating glacier the terminus wedge is the wrong picture, and the chapter's final section replaces it with two reservoirs, the interior thickness $H$ and the grounding-line position $L$. Mass conservation gives

$$ \frac{dL}{dt} = \frac{Q - Q_g}{h_g}, \qquad \frac{dH}{dt} = P - \frac{Q_g}{L} - \frac{H}{h_g L}\left(Q - Q_g\right), $$

with interior flux $Q = \nu H^{\alpha}/L^{\gamma}$ (Weertman sliding gives $\alpha = 7$, $\gamma = 3$, $\nu = (\rho_i g/C)^n$), grounding-line flux $Q_g = \Omega h_g^{\beta}$, and the flotation thickness tied to the bed depth, $h_g = -\lambda\, b(L)$ with $\lambda = \rho_w/\rho_i$. We use the Schoof flux exponent $\beta = (m+n+3)/(m+1) = 4.75$ and the parameter values of Robel, Roe, and Haseloff's flowline benchmark, all set up in the next cell.

Linearized about a steady state, the system has a fast timescale (decades to centuries, grounding-line adjustment) and a slow one (millennia, interior relaxation), and the slow mode's stability hinges on the sign of

$$ S_T = 1 + \frac{\beta\,\lambda\,\bar b_x\,\bar L}{\bar h_g}, $$

where $\bar b_x$ is the bed slope at the grounding line, negative where the bed deepens seaward (prograde) and positive where it deepens inland (retrograde). Negative $S_T$ means stable, and a sufficiently retrograde bed flips the sign. That is the marine ice-sheet instability, arrived at with no stress balance anywhere in sight.

**Task 5 (optional):** Complete the integrator below, then run three experiments. First, spin up on the prograde bed from $H = 2000$ m, $L = 400$ km for 30,000 years and record the equilibrium $\bar H$, $\bar L$, $\bar h_g$. Second, drop $P$ by 5% and watch the fast-then-slow adjustment, checking the new equilibrium against the prediction $L'/\bar L = -(1/S_T)(P'/\bar P)$. Third, transplant the equilibrium state onto the retrograde bed, nudge $L$ by $\pm 1$ km, and run both trajectories. Compute $S_T$ for both beds and connect its sign to what you see.

In [ ]:
SPY = 3.15576e7   # seconds per year

rho_i, rho_w, grav = 917.0, 1028.0, 9.81
lam = rho_w / rho_i
n, m = 3.0, 1.0 / 3.0
A_g = 4.22e-25      # flow-law coefficient, Pa^-n s^-1
C = 7.624e6         # basal friction coefficient, Pa m^-1/3 s^1/3
theta = 0.6         # buttressing parameter
P_bar = 0.3 / SPY   # accumulation, m/s

nu = (rho_i * grav / C)**n
beta = (m + n + 3.0) / (m + 1.0)
Omega = (A_g * (rho_i * grav)**(n + 1)
         * (theta * (1.0 - 1.0 / lam))**n / (4.0**n * C))**(1.0 / (m + 1.0))


def bed_prograde(x):
    return -100.0 - 1.0e-3 * x      # deepens seaward, b_x = -1e-3


def bed_retrograde(x):
    return -900.0 + 1.0e-3 * x      # deepens inland, b_x = +1e-3


def Q_interior(H, L):
    return nu * H**(2 * n + 1) / L**n          # m^2/s


def Q_grounding(h_g):
    return Omega * h_g**beta                   # m^2/s


def run_robel(bed, H0, L0, P, years, dt_yr=1.0):
    """Integrate the two-stage marine model with forward Euler.

    Returns time (yr), H (m), L (m) arrays.
    """
    nsteps = int(years / dt_yr)
    dt = dt_yr * SPY
    t = np.arange(nsteps + 1) * dt_yr
    H = np.zeros(nsteps + 1)
    L = np.zeros(nsteps + 1)
    H[0], L[0] = H0, L0
    for i in range(nsteps):
        h_g = -lam * bed(L[i])
        Q = Q_interior(H[i], L[i])
        Qg = Q_grounding(h_g)
        # YOUR CODE HERE: dLdt and dHdt from the two model equations,
        # then the Euler updates L[i+1] and H[i+1]
        pass
    return t, H, L


# Experiment 1: spin-up on the prograde bed
# t1, H1, L1 = run_robel(bed_prograde, 2000.0, 400.0e3, P_bar, 30000.0)

# Experiment 2: 5% drop in accumulation from the spun-up state
# t2, H2, L2 = run_robel(bed_prograde, H1[-1], L1[-1], 0.95 * P_bar, 30000.0)

# Experiment 3: same state transplanted to the retrograde bed, nudged +/- 1 km
# t3a, H3a, L3a = run_robel(bed_retrograde, H1[-1], L1[-1] + 1000.0, P_bar, 5000.0)
# t3b, H3b, L3b = run_robel(bed_retrograde, H1[-1], L1[-1] - 1000.0, P_bar, 5000.0)

# YOUR CODE HERE: plot L(t) for each experiment, compute S_T for both
# beds at the spun-up equilibrium, and compare the experiment-2 retreat
# with the prediction L'/L = -(1/S_T) * (P'/P)

## Synthesis

Write a short paragraph drawing the lab together, addressing the points below.

1. The response time $\tau$ is an e-folding scale, not a delay, and the three-stage model shows the difference. Explain why fitting a simple exponential to an observed terminus record would overestimate the glacier's memory, and why the distinction matters when an observed retreat is tested against the null hypothesis of noise-driven wandering, as in the attribution work of Roe, Baker, and Herla.
2. Your Task 3 wanderings and your Task 4 committed retreat are two faces of the same filter. State, in one or two sentences, what each implies about how a single glacier length record should and should not be read.
3. If you ran the extension, the two-stage marine model produced the marine ice-sheet instability from nothing but mass conservation and a flux rule. What does that tell you about what the instability fundamentally is, and why does the millennial slow timescale $T_S$ mean that a marine glacier which has looked stable for centuries may be telling you nothing of the kind?